In [7]:
import requests, time, math

ACIS = "https://data.rcc-acis.org"

TARGETS = {
    "Guthrie": (35.879, -97.425),
    "Idabel":  (33.894, -94.826),
}
BOX_PAD = 0.6
START, END = "1991-01-01", "2020-12-31"   
COMPLETENESS = 0.80

NET = {"1": "WBAN", "2": "COOP", "3": "FAA", "4": "WMO", "5": "ICAO",
       "6": "GHCN", "7": "ThreadEx", "8": "CoCoRaHS", "9": "Misc",
       "10": "AWDN", "16": "RAWS"}

def total_days(start, end):
    import datetime as dt
    s = dt.date(*map(int, start.split("-")))
    e = dt.date(*map(int, end.split("-")))
    return (e - s).days + 1

TOTAL_DAYS = total_days(START, END)

def bbox(lat, lon, pad=BOX_PAD):
    return f"{lon-pad},{lat-pad},{lon+pad},{lat+pad}"   # W,S,E,N

def miles(lat1, lon1, lat2, lon2):
    R = 3959.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def find_stations(lat, lon):
    payload = {
        "bbox": bbox(lat, lon),
        "meta": "name,sids,ll,valid_daterange",
        "elems": ["maxt", "mint", "pcpn"],   # all-network return
    }
    r = requests.post(f"{ACIS}/StnMeta", json=payload, timeout=60)
    r.raise_for_status()
    return r.json().get("meta", [])

def pick_sid(sids):
    parsed = [(p[0], p[1]) for p in (s.split() for s in sids) if len(p) == 2]
    for want in ("2", "6", "1"):
        for ident, net in parsed:
            if net == want:
                return f"{ident} {net}", NET.get(net, net)
    if parsed:
        ident, net = parsed[0]
        return f"{ident} {net}", NET.get(net, net)
    return None, None

def por_span(meta):
    yrs = []
    for pair in meta.get("valid_daterange", []):
        if pair and pair[0] and pair[1]:
            yrs += [pair[0][:4], pair[1][:4]]
    return f"{min(yrs)}-{max(yrs)}" if yrs else "  n/a  "

def availability(sid):
    payload = {
        "sid": sid, "sdate": START, "edate": END,
        "elems": [{"name": n, "interval": "dly"} for n in ("maxt", "mint", "pcpn")],
    }
    try:
        r = requests.post(f"{ACIS}/StnData", json=payload, timeout=120)
        r.raise_for_status()
        data = r.json().get("data", [])
    except Exception:
        return None
    if not data:
        return None
    miss = ("M", "", None, "-9999")
    c = {"maxt": 0, "mint": 0, "tmean": 0, "pcpn": 0}
    for row in data:
        mx, mn, pc = row[1], row[2], row[3]
        if mx not in miss: c["maxt"] += 1
        if mn not in miss: c["mint"] += 1
        if pc not in miss: c["pcpn"] += 1
        if mx not in miss and mn not in miss: c["tmean"] += 1   # Tmean = (Tmax+Tmin)/2
    return c

for place, (lat, lon) in TARGETS.items():
    print(f"\n=== ALL sites near {place}  |  {START[:4]}-{END[:4]}  |  sorted by distance ===")
    print(f"  {'TAG':4s}  {'STATION':32s} {'SID':14s} {'NET':9s} {'DIST':6s} "
          f"{'POR':11s} {'Tmax':6s} {'Tmin':6s} {'Tmean':6s} {'Precip':6s}")
    rows, seen = [], set()
    for st in find_stations(lat, lon):
        sid, net = pick_sid(st.get("sids", []))
        if not sid or sid in seen:
            continue
        seen.add(sid)
        name = st.get("name", "?")
        try:
            slat, slon = st["ll"][1], st["ll"][0]
            dist = miles(lat, lon, slat, slon)
        except Exception:
            dist = float("nan")
        por = por_span(st)
        c = availability(sid)
        time.sleep(0.25)
        rows.append((dist, name, sid, net, por, c))

    rows.sort(key=lambda x: (math.isnan(x[0]), x[0]))
    for dist, name, sid, net, por, c in rows:
        d = f"{dist:4.0f}mi" if not math.isnan(dist) else "  ?mi"
        if c is None:
            print(f"  n/a   {name:32s} {sid:14s} {net:9s} {d:6s} {por:11s}  (no data)")
            continue
        p = {k: c[k] / TOTAL_DAYS for k in c}
        keep = all(p[k] >= COMPLETENESS for k in ("maxt", "mint", "tmean", "pcpn"))
        print(f"  {'KEEP' if keep else 'DROP'}  {name:32s} {sid:14s} {net:9s} {d:6s} "
              f"{por:11s} {p['maxt']:5.0%} {p['mint']:5.0%} {p['tmean']:5.0%} {p['pcpn']:5.0%}")

print(f"\n  Window: {START} to {END} ({TOTAL_DAYS} days).  Tmean = (Tmax+Tmin)/2 availability.")
print(f"  KEEP = >= {int(COMPLETENESS*100)}% on Tmax, Tmin, Tmean, and Precip.")


=== ALL sites near Guthrie  |  1991-2020  |  sorted by distance ===
  TAG   STATION                          SID            NET       DIST   POR         Tmax   Tmin   Tmean  Precip
  DROP  GUTHRIE 3.0 N                    US1OKLG0008 6  GHCN         0mi 2008-2012      0%    0%    0%    5%
  DROP  GUTHRIE 2.9 N                    US1OKLG0011 6  GHCN         0mi 2012-2017      0%    0%    0%   16%
  DROP  GUTHRIE 3.7 NNW                  US1OKLG0006 6  GHCN         1mi 2007-2019      0%    0%    0%   40%
  DROP  GUTHRIE MUNICIPAL AP             343818 2       COOP         2mi 1998-2026     75%   75%   75%   75%
  DROP  GUTHRIE 4WSW MESONET             343820 2       COOP         4mi 2000-2026     40%   40%   40%   40%
  DROP  GUTHRIE 4.3 ENE                  US1OKLG0020 6  GHCN         4mi 2019-2026      0%    0%    0%    6%
  DROP  GUTHRIE 1.9 S                    US1OKLG0002 6  GHCN         5mi 2006-2017      0%    0%    0%   33%
  DROP  GUTHRIE 5S                       343821 2      

In [8]:
import requests, time, math, datetime as dt

ACIS = "https://data.rcc-acis.org"

TARGETS = {
    "Guthrie": (35.879, -97.425),
    "Idabel":  (33.894, -94.826),
}
BOX_PAD = 0.6
START, END = "1991-01-01", "2020-12-31"     # your window
COMPLETENESS = 0.80

NET = {"1": "WBAN", "2": "COOP", "3": "FAA", "4": "WMO", "5": "ICAO",
       "6": "GHCN", "7": "ThreadEx", "8": "CoCoRaHS", "9": "Misc",
       "10": "AWDN", "16": "RAWS"}

def total_days(a, b):
    s = dt.date(*map(int, a.split("-"))); e = dt.date(*map(int, b.split("-")))
    return (e - s).days + 1
TOTAL_DAYS = total_days(START, END)

def bbox(lat, lon, pad=BOX_PAD):
    return f"{lon-pad},{lat-pad},{lon+pad},{lat+pad}"

def miles(lat1, lon1, lat2, lon2):
    R = 3959.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def find_stations(lat, lon):
    payload = {"bbox": bbox(lat, lon), "meta": "name,sids,ll",
               "elems": ["maxt", "mint", "pcpn"]}
    r = requests.post(f"{ACIS}/StnMeta", json=payload, timeout=60)
    r.raise_for_status()
    return r.json().get("meta", [])

def pick_sid(sids):
    parsed = [(p[0], p[1]) for p in (s.split() for s in sids) if len(p) == 2]
    for want in ("2", "6", "1"):
        for ident, net in parsed:
            if net == want:
                return f"{ident} {net}", NET.get(net, net)
    if parsed:
        ident, net = parsed[0]
        return f"{ident} {net}", NET.get(net, net)
    return None, None

def station_data(sid):
    """Pull daily maxt, mint, pcpn for the window. Returns list of (date, mx, mn, pc)."""
    payload = {"sid": sid, "sdate": START, "edate": END,
               "elems": [{"name": n, "interval": "dly"} for n in ("maxt", "mint", "pcpn")]}
    try:
        r = requests.post(f"{ACIS}/StnData", json=payload, timeout=120)
        r.raise_for_status()
        return r.json().get("data", [])
    except Exception:
        return None

MISS = ("M", "", None, "-9999")

def analyze(data):
    """Return counts and the in-window coverage span (first & last year with ANY temp/precip)."""
    c = {"maxt": 0, "mint": 0, "tmean": 0, "pcpn": 0}
    first_yr = last_yr = None
    for row in data:
        date, mx, mn, pc = row[0], row[1], row[2], row[3]
        has_any = False
        if mx not in MISS: c["maxt"] += 1; has_any = True
        if mn not in MISS: c["mint"] += 1; has_any = True
        if pc not in MISS: c["pcpn"] += 1; has_any = True
        if mx not in MISS and mn not in MISS: c["tmean"] += 1
        if has_any:
            yr = date[:4]
            if first_yr is None: first_yr = yr
            last_yr = yr
    span = f"{first_yr}-{last_yr}" if first_yr else "  none "
    return c, span

for place, (lat, lon) in TARGETS.items():
    print(f"\n=== ALL sites near {place}  |  {START[:4]}-{END[:4]}  |  sorted by distance ===")
    print(f"  {'TAG':4s}  {'STATION':32s} {'SID':14s} {'NET':6s} {'DIST':6s} "
          f"{'COVERED':11s} {'Tmax':6s} {'Tmin':6s} {'Tmean':6s} {'Precip':6s}")
    rows, seen = [], set()
    for st in find_stations(lat, lon):
        sid, net = pick_sid(st.get("sids", []))
        if not sid or sid in seen:
            continue
        seen.add(sid)
        name = st.get("name", "?")
        try:
            slat, slon = st["ll"][1], st["ll"][0]
            dist = miles(lat, lon, slat, slon)
        except Exception:
            dist = float("nan")
        data = station_data(sid)
        time.sleep(0.25)
        rows.append((dist, name, sid, net, data))

    rows.sort(key=lambda x: (math.isnan(x[0]), x[0]))
    for dist, name, sid, net, data in rows:
        d = f"{dist:4.0f}mi" if not math.isnan(dist) else "  ?mi"
        if not data:
            print(f"  n/a   {name:32s} {sid:14s} {net:6s} {d:6s} {'none':11s}  (no data)")
            continue
        c, span = analyze(data)
        p = {k: c[k] / TOTAL_DAYS for k in c}
        keep = all(p[k] >= COMPLETENESS for k in ("maxt", "mint", "tmean", "pcpn"))
        print(f"  {'KEEP' if keep else 'DROP'}  {name:32s} {sid:14s} {net:6s} {d:6s} "
              f"{span:11s} {p['maxt']:5.0%} {p['mint']:5.0%} {p['tmean']:5.0%} {p['pcpn']:5.0%}")

print(f"\n  Window: {START} to {END} ({TOTAL_DAYS} days).")
print(f"  COVERED = first-last year WITHIN the window that has data (not the station's lifetime).")
print(f"  Tmean = (Tmax+Tmin)/2 availability.  KEEP = >= {int(COMPLETENESS*100)}% on all four.")


=== ALL sites near Guthrie  |  1991-2020  |  sorted by distance ===
  TAG   STATION                          SID            NET    DIST   COVERED     Tmax   Tmin   Tmean  Precip
  DROP  GUTHRIE 3.0 N                    US1OKLG0008 6  GHCN      0mi 2008-2012      0%    0%    0%    5%
  DROP  GUTHRIE 2.9 N                    US1OKLG0011 6  GHCN      0mi 2012-2017      0%    0%    0%   16%
  DROP  GUTHRIE 3.7 NNW                  US1OKLG0006 6  GHCN      1mi 2007-2019      0%    0%    0%   40%
  DROP  GUTHRIE MUNICIPAL AP             343818 2       COOP      2mi 1998-2020     75%   75%   75%   75%
  DROP  GUTHRIE 4WSW MESONET             343820 2       COOP      4mi 2000-2020     40%   40%   40%   40%
  DROP  GUTHRIE 4.3 ENE                  US1OKLG0020 6  GHCN      4mi 2019-2020      0%    0%    0%    6%
  DROP  GUTHRIE 1.9 S                    US1OKLG0002 6  GHCN      5mi 2006-2017      0%    0%    0%   33%
  DROP  GUTHRIE 5S                       343821 2       COOP      5mi 1991-2015

In [9]:
import csv

for place, (lat, lon) in TARGETS.items():
    print(f"\n=== ALL sites near {place}  |  {START[:4]}-{END[:4]}  |  sorted by distance ===")
    rows, seen = [], set()
    for st in find_stations(lat, lon):
        sid, net = pick_sid(st.get("sids", []))
        if not sid or sid in seen:
            continue
        seen.add(sid)
        name = st.get("name", "?")
        try:
            slat, slon = st["ll"][1], st["ll"][0]
            dist = miles(lat, lon, slat, slon)
        except Exception:
            dist = float("nan")
        data = station_data(sid)
        time.sleep(0.25)
        rows.append((dist, name, sid, net, data))

    rows.sort(key=lambda x: (math.isnan(x[0]), x[0]))

    out_rows = []
    for dist, name, sid, net, data in rows:
        if not data:
            out_rows.append([name, sid, net, round(dist, 1), "none", "", "", "", "", "DROP"])
            continue
        c, span = analyze(data)
        p = {k: round(100 * c[k] / TOTAL_DAYS) for k in c}   # whole-number percents
        keep = all(c[k] / TOTAL_DAYS >= COMPLETENESS for k in ("maxt", "mint", "tmean", "pcpn"))
        out_rows.append([name, sid, net, round(dist, 1), span,
                         p["maxt"], p["mint"], p["tmean"], p["pcpn"],
                         "KEEP" if keep else "DROP"])
        print(f"  {'KEEP' if keep else 'DROP'}  {name:32s} {sid:14s} {net:6s} "
              f"{dist:4.0f}mi {span:11s} {p['maxt']:3d}% {p['mint']:3d}% {p['tmean']:3d}% {p['pcpn']:3d}%")

    fname = f"stations_{place}_{START[:4]}_{END[:4]}.csv"
    with open(fname, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Station", "SID", "Network", "Distance_mi", "Covered_years",
                    "Tmax_%", "Tmin_%", "Tmean_%", "Precip_%", "Keep"])
        w.writerows(out_rows)
    print(f"  -> wrote {fname} ({len(out_rows)} stations)")


=== ALL sites near Guthrie  |  1991-2020  |  sorted by distance ===
  DROP  GUTHRIE 3.0 N                    US1OKLG0008 6  GHCN      0mi 2008-2012     0%   0%   0%   5%
  DROP  GUTHRIE 2.9 N                    US1OKLG0011 6  GHCN      0mi 2012-2017     0%   0%   0%  16%
  DROP  GUTHRIE 3.7 NNW                  US1OKLG0006 6  GHCN      1mi 2007-2019     0%   0%   0%  40%
  DROP  GUTHRIE MUNICIPAL AP             343818 2       COOP      2mi 1998-2020    75%  75%  75%  75%
  DROP  GUTHRIE 4WSW MESONET             343820 2       COOP      4mi 2000-2020    40%  40%  40%  40%
  DROP  GUTHRIE 4.3 ENE                  US1OKLG0020 6  GHCN      4mi 2019-2020     0%   0%   0%   6%
  DROP  GUTHRIE 1.9 S                    US1OKLG0002 6  GHCN      5mi 2006-2017     0%   0%   0%  33%
  DROP  GUTHRIE 5S                       343821 2       COOP      5mi 1991-2015    78%  77%  77%  80%
  DROP  GUTHRIE SCS                      343830 2       COOP      5mi   none        0%   0%   0%   0%
  DROP  GUTHR